## 1. Import libraries

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

## 2. Set up environments

In [ ]:
class SpaceRepetitionEnv(gym.Env):
  """
  Gymnasium environment for RL-based spaced repetition scheduling.

  State  : (N, 3) matrix flattened → [R_0, k_0, Δt_0, R_1, k_1, Δt_1, ...]
  Action : integer i ∈ {0, ..., N-1}  (chapter to study today)
  Reward : mean retention across all N chapters after transition
  """
  def __init__(
      self,
      N: int = 5,
      T: int = 30,
      h0: float = 1.0,
      s: float = 2.0
  ):
    super.__init__()
    self.N = N                     # number of chapters
    self.T = T                     # episode length (days)
    self.h0 = h0                   # base half-life
    self.s = s                     # retention growth factor
    self.h_max = T / 2.0           # half-life upper bound
    
    # --- Spaces ---
    # Observation: [R_i, k_i, Δt_i] for each chapter i
    # R_i  ∈ [0, 1]
    # k_i  ∈ [0, T]     (at most T studies in T days)
    # Δt_i ∈ [0, T]
    low  = np.zeros(N * 3, dtype=np.float32)
    high = np.tile([1.0, float(T), float(T)], N).astype(np.float32)

    self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)
    self.action_space      = spaces.Discrete(N)

    # --- Internal state (initialized in reset) ---
    self._retention_arr: np.ndarray  # shape (N,)  — R_i
    self._k_arr:         np.ndarray  # shape (N,)  — k_i  (int)
    self._delta_t_arr:   np.ndarray  # shape (N,)  — Δt_i (int)
    self._day:           int

    # ------------------------------------------------------------------
    # Core helpers
    # ------------------------------------------------------------------

    def _half_life(self, k: float) -> float:
      """h(k) = min(h0 * s^sqrt(k), h_max)"""
      if k == 0:
        return 0.0
      return float(np.clip(self.h0 * (self.s ** np.sqrt(k)), 0.0, self.h_max))

    def _retention(self, k: float, delta_t: float) -> float:
      """R = 2^(-Δt / h(k));  R = 0 if never studied."""
      if k == 0:
        return 0.0
      h = self._half_life(k)
      return float(2.0 ** (-delta_t / h))

    def _recompute_retentions(self) -> None:
      """Recompute R_i for all chapters from current k, Δt arrays."""
      for i in range(self.N):
        self._retention_arr[i] = self._retention(
          self._k_arr[i], self._delta_t_arr[i]
        )

## 3. DQN agent